# Categorical data

Categoricals are a pandas data type corresponding to categorical variables in statistics. A categorical variable takes on a limited, and usually fixed, number of possible values. 

Examples are gender, social class, blood type, country affiliation, observation time or rating via Likert scales.

In contrast to statistical categorical variables, categorical data might have an order (e.g. ‘strongly agree’ vs ‘agree’ or ‘first observation’ vs. ‘second observation’), but numerical operations (additions, divisions, …) are not possible.

The categorical data type is useful in the following cases:

- A string variable consisting of only a few different values. Converting such a string variable to a categorical variable will save some memory, see here.

- The lexical order of a variable is not the same as the logical order (“one”, “two”, “three”). By converting to a categorical and specifying an order on the categories, sorting and min/max will use the logical order instead of the lexical order, see here.

- As a signal to other Python libraries that this column should be treated as a categorical variable (e.g. to use suitable statistical methods or plot types).

## Object Creation

By specifying dtype="category" when constructing a Series:

In [1]:
import pandas as pd 
import numpy as np

In [2]:
s = pd.Series(["a", "b", "c", "a"], dtype="category")
s

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [3]:
s.values

['a', 'b', 'c', 'a']
Categories (3, object): ['a', 'b', 'c']

In [5]:
s.values.categories

Index(['a', 'b', 'c'], dtype='object')

In [34]:
# 목록 중 몇 번째에 해당하는가?
s.values.codes

array([0, 1, 2, 0], dtype=int8)

The category codes of this categorical index.

Codes are an array of integers which are the positions of the actual values in the categories array.

There is no setter, use the other categorical methods and the normal item setter to change values in the categorical.

By using special functions, such as cut(), which groups data into discrete bins.

In [6]:
df = pd.DataFrame({"value": np.random.randint(0, 100, 20)})
df

,value
0,9
1,99
2,40
3,90
4,99
5,38
6,53
7,20
8,25
9,48


In [9]:
label1 = ["{0} - {1}".format(i, i + 9) for i in range(0, 100, 10)]
label1

['0 - 9',
 '10 - 19',
 '20 - 29',
 '30 - 39',
 '40 - 49',
 '50 - 59',
 '60 - 69',
 '70 - 79',
 '80 - 89',
 '90 - 99']

In [10]:
df["group"] = pd.cut(df.value, range(0, 105, 10), right=False, labels=label1)
df
# df.value: 구간을 나눌 대상이 되는 수치형 데이터(Series)입니다.
# range(0, 105, 10): 데이터를 나눌 기준점, [0, 10, 20, 30, ..., 90, 100]
# 즉, 0~10, 10~20 ... 90~100 구간이 만들어집니다.
# labels=label1: 그냥 이름만
# 실제로는 9 = 0이상 10미만 그룹에 속하기 때문에 label1[0]이 들어가서 0-9가 뜸


,value,group
0,9,0 - 9
1,99,90 - 99
2,40,40 - 49
3,90,90 - 99
4,99,90 - 99
5,38,30 - 39
6,53,50 - 59
7,20,20 - 29
8,25,20 - 29
9,48,40 - 49


Another way of creating category 

By passing a pandas.Categorical object to a Series or assigning it to a DataFrame.

In [25]:
raw_cat = pd.Categorical(
    ["a", "b", "c", "a"], categories=["b", "c", "d"], ordered=False
)
raw_cat
# raw_cat.min() 이건 안됨 order가 없으니까

[NaN, 'b', 'c', NaN]
Categories (3, object): ['b', 'c', 'd']

In [12]:
s = pd.Series(raw_cat)
s

0    NaN
1      b
2      c
3    NaN
dtype: category
Categories (3, object): ['b', 'c', 'd']

In [14]:
raw_cat.codes
# NaN: -1값으로

array([-1,  0,  1, -1], dtype=int8)

more examples

In [15]:
c = pd.Categorical(['a', 'b', 'c', 'a', 'b', 'c'], ordered=True,
                   categories=['c', 'b', 'a'])
c


['a', 'b', 'c', 'a', 'b', 'c']
Categories (3, object): ['c' < 'b' < 'a']

In [22]:
c.codes

array([2, 1, 0, 2, 1, 0], dtype=int8)

In [23]:
c.min()

'c'

In [26]:
df = pd.DataFrame({"A": ["a", "b", "c", "a"]})
df

,A
0,a
1,b
2,c
3,a


In [27]:
df["B"] = raw_cat

In [28]:
df

,A,B
0,a,NaN
1,b,b
2,c,c
3,a,NaN


dataframe creation

In [29]:
df = pd.DataFrame({"A": list("abca"), "B": list("bccd")}, dtype="category")
df

,A,B
0,a,b
1,b,c
2,c,c
3,a,d


In [30]:
df.dtypes

A    category
B    category
dtype: object

Note that the categories present in each column differ; the conversion is done column by column, so only labels present in a given column are categories:

In [31]:
df["A"]

0    a
1    b
2    c
3    a
Name: A, dtype: category
Categories (3, object): ['a', 'b', 'c']

In [32]:
df["B"]

0    b
1    c
2    c
3    d
Name: B, dtype: category
Categories (3, object): ['b', 'c', 'd']

## Controlling behavior

In the examples above where we passed dtype='category', we used the default behavior:

1. Categories are inferred from the data.

2. Categories are **unordered**.

To control those behaviors, instead of passing 'category', use an instance of CategoricalDtype

In [36]:
from pandas.api.types import CategoricalDtype
'''
기존에 pd.Categorical()을 사용할 때는 데이터를 만들면서 동시에 카테고리 순서와 종류를 정의했다면, 
CategoricalDtype을 사용하면 "이런 규칙을 가진 타입을 쓰겠다"라고 먼저 선언한 뒤 
여러 데이터 프레임이나 컬럼에 재사용할 수 있습니다.
df.astype() 메서드와 결합하여 기존 컬럼의 타입을 한 번에 변경하기 좋습니다.
'''


'\n기존에 pd.Categorical()을 사용할 때는 데이터를 만들면서 동시에 카테고리 순서와 종류를 정의했다면, \nCategoricalDtype을 사용하면 "이런 규칙을 가진 타입을 쓰겠다"라고 먼저 선언한 뒤 \n여러 데이터 프레임이나 컬럼에 재사용할 수 있습니다.\ndf.astype() 메서드와 결합하여 기존 컬럼의 타입을 한 번에 변경하기 좋습니다.\n'

In [37]:
s = pd.Series(["a", "b", "c", "a"])
s

0    a
1    b
2    c
3    a
dtype: object

In [38]:
cat_type = CategoricalDtype(categories=["b", "c", "d"], ordered=True)
cat_type

CategoricalDtype(categories=['b', 'c', 'd'], ordered=True, categories_dtype=object)

In [39]:
s_cat = s.astype(cat_type)
s_cat

0    NaN
1      b
2      c
3    NaN
dtype: category
Categories (3, object): ['b' < 'c' < 'd']

Similarly, a CategoricalDtype can be used with a DataFrame to ensure that categories are consistent among all columns.

In [40]:
df = pd.DataFrame({"A": list("abca"), "B": list("bccd")})
df

,A,B
0,a,b
1,b,c
2,c,c
3,a,d


In [41]:
cat_type = CategoricalDtype(categories=list("abcd"), ordered=True)

In [42]:
df_cat = df.astype(cat_type)

In [43]:
df_cat["A"]

0    a
1    b
2    c
3    a
Name: A, dtype: category
Categories (4, object): ['a' < 'b' < 'c' < 'd']

In [44]:
df_cat["B"]

0    b
1    c
2    c
3    d
Name: B, dtype: category
Categories (4, object): ['a' < 'b' < 'c' < 'd']

To get back to the original Series or NumPy array, use Series.astype(original_dtype) or np.asarray(categorical):  
카테고리형(Categorical)으로 변환된 데이터를 다시 원래의 일반적인 데이터 형태(문자열, 숫자 등)로 되돌림  
Pandas의 Series 객체에서 다시 일반 타입(예: str, int, object)으로 바꾸고 싶을 때 사용합니다.  
카테고리 객체를 순수한 NumPy 배열(ndarray) 형태로 뽑아내고 싶을 때 사용합니다.

In [45]:
s = pd.Series(["a", "b", "c", "a"])
s2 = s.astype("category")
s2

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [46]:
s2.astype(str)

0    a
1    b
2    c
3    a
dtype: object

## CategoricalDtype


A categorical’s type is fully described by

1. categories: a sequence of unique values and no missing values

2. ordered: a boolean

In [53]:
# CategoticalDtype에서는 중복 허용 안됨
CategoricalDtype(["a", "b", "c"])

CategoricalDtype(categories=['a', 'b', 'c'], ordered=False, categories_dtype=object)

In [48]:
CategoricalDtype(["a", "b", "c"], ordered=True)

CategoricalDtype(categories=['a', 'b', 'c'], ordered=True, categories_dtype=object)

Two instances of CategoricalDtype compare equal whenever they have the same categories and order. When comparing two unordered categoricals, the order of the categories is not considered.

In [52]:
c1 = CategoricalDtype(["a", "b", "c"], ordered=False)
c1 == CategoricalDtype(["b", "c", "a"], ordered=False)
# 순서가 없는 경우에는 내용물이 같은지만 따짐

True

In [55]:
c1 = CategoricalDtype(["a", "b", "c"], ordered=False)
c1 == CategoricalDtype(["b", "c", "a"], ordered=True)
# 두 타입이 ==에서 True가 나오려면 다음 세 가지가 모두 충족되어야 합니다.
# 구성 요소(Categories)가 같아야 함.
# 순서 여부(ordered)가 같아야 함.
# (ordered=True인 경우) 구성 요소의 배치 순서까지 같아야 함.
# 지금 케이스는 2번 조건(False vs True)에서 어긋났기 때문에 결과는 False입니다.

False

In [56]:
s = pd.Series(pd.Categorical(["a", "b", "c", "a"], ordered=False))
s

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [57]:
s.sort_values() # false이면 사전식 정렬
# s.min(), s.max() 당연히 안됨

0    a
3    a
1    b
2    c
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [67]:
s = pd.Series(["a", "b", "c", "a"]).astype(CategoricalDtype(ordered=False))
s

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [68]:
s.sort_values()

0    a
3    a
1    b
2    c
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [69]:
s.cat.as_ordered()

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a' < 'b' < 'c']

In [71]:
s.cat.as_unordered()

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [72]:
s

0    a
1    b
2    c
3    a
dtype: category
Categories (3, object): ['a', 'b', 'c']

In [74]:
s = pd.Series([1, 2, 3, 1], dtype="category")
s

0    1
1    2
2    3
3    1
dtype: category
Categories (3, int64): [1, 2, 3]

In [76]:
s = s.cat.set_categories([2, 3, 1], ordered=True)
s

0    1
1    2
2    3
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [77]:
s = s.sort_values()
s

1    2
2    3
0    1
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

In [78]:
s.min()

np.int64(2)

### reordering

In [80]:
s = pd.Series([1, 2, 3, 1], dtype="category")
s

0    1
1    2
2    3
3    1
dtype: category
Categories (3, int64): [1, 2, 3]

In [82]:
s = s.cat.reorder_categories([2, 3, 1], ordered=True)
s

0    1
1    2
2    3
3    1
dtype: category
Categories (3, int64): [2 < 3 < 1]

# Multi column sorting


In [83]:
dfs = pd.DataFrame(
    {
        "A": pd.Categorical(
            list("bbeebbaa"),
            categories=["e", "a", "b"],
            ordered=True,
        ),
        "B": [1, 2, 1, 2, 2, 1, 2, 1],
    }
)
dfs

,A,B
0,b,1
1,b,2
2,e,1
3,e,2
4,b,2
5,b,1
6,a,2
7,a,1


In [84]:
dfs.sort_values(by=["A", "B"])
# A가 1순위, B가 2순위, B는 순서가 없으므로 오름차순

,A,B
2,e,1
3,e,2
7,a,1
6,a,2
0,b,1
5,b,1
1,b,2
4,b,2


In [85]:
dfs["A"] = dfs["A"].cat.reorder_categories(["a", "b", "e"])
dfs.sort_values(by=["A", "B"])

,A,B
7,a,1
6,a,2
0,b,1
5,b,1
1,b,2
4,b,2
2,e,1
3,e,2


In [86]:
dfs.sort_values(by='B') #A는 원래 있던 순서대로 배치

,A,B
0,b,1
2,e,1
7,a,1
5,b,1
3,e,2
1,b,2
4,b,2
6,a,2
